# 01 — Data Pipeline: Generate Contrastive Triplets

This notebook generates the training data for the HCCS scorer.

**What it produces:** `data/triplets.jsonl` — ~1,500–3,000 contrastive triplets  
of the form `(query, positive_context, negative_context, hallucination_type)`.

**How it works:**
1. Load tasks from CodeHaluEval (HuggingFace: `Yuchen111/CodeHaluEval`)
2. For each task, sample 6 random context subsets from the reference solutions
3. Generate code with each subset using DeepSeek-Coder-1.3B
4. Execute each generated code against the task's test cases
5. Pair a PASSING subset with a FAILING subset → contrastive triplet
6. Save triplets incrementally to Drive (Colab-safe)

**Runtime:** ~10 hours on T4 GPU (~2 CU)

---
All cells are **idempotent** — safe to re-run after disconnect.

## Setup

In [1]:
!pip install torch transformers datasets rank-bm25 tqdm --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import sys, os
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/HaluGuard'
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
!pip install -e '.[dev]' -q

from notebooks.utils import check_gpu, print_env_summary, get_drive_path, append_jsonl, count_jsonl
DEVICE = check_gpu()
print_env_summary()

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from pathlib import Path
from tqdm import tqdm

from haluguard.data_pipeline import (
    ContrastiveTriplet,
    create_triplets_from_task,
    load_triplets,
    save_triplets,
    summarise_triplets,
)

DRIVE_DIR    = get_drive_path('HaluGuard/data')
TRIPLETS_PATH = DRIVE_DIR / 'triplets.jsonl'
print(f'Triplets path: {TRIPLETS_PATH}')
print(f'Existing triplets: {count_jsonl(TRIPLETS_PATH)}')

## 1. Load and Inspect Dataset

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
import json

# load_dataset('Yuchen111/CodeHaluEval') fails due to mixed column types in the
# raw JSON (Arrow can't reconcile string vs list in the 'input' column).
# We bypass Arrow entirely by downloading and parsing the raw JSON files.

repo_id  = 'Yuchen111/CodeHaluEval'
all_files = list(list_repo_files(repo_id, repo_type='dataset'))
json_files = [f for f in all_files if f.endswith('.json')]
print(f'Found {len(json_files)} JSON files:', json_files)

all_samples = []
for fname in json_files:
    path = hf_hub_download(repo_id, fname, repo_type='dataset')
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, list):
        all_samples.extend(data)
    elif isinstance(data, dict):
        all_samples.append(data)

# Normalise: ensure 'input' is always a string
for s in all_samples:
    if isinstance(s.get('input'), list):
        s['input'] = '\n'.join(str(x) for x in s['input'])
    elif s.get('input') is None:
        s['input'] = ''
    if s.get('output') is None:
        s['output'] = ''

print(f'Total samples loaded: {len(all_samples)}')
print(f'Columns: {list(all_samples[0].keys())}')

# Wrap in a simple list — use like: for sample in all_samples[:N_TASKS]
ds = all_samples   # drop-in replacement for HF Dataset object

In [ ]:
# IMPORTANT: Run this cell first to see the actual column names.
# The field names below (query, repo_files, test_code, task_id) are placeholders
# — update FIELD_MAP in the next cell to match what you see here.
sample = ds[0]
for key, val in sample.items():
    preview = str(val)[:150] + '...' if len(str(val)) > 150 else str(val)
    print(f'{key!r:25s}: {preview}')

In [ ]:
import ast
from haluguard.hccs import HallucinationType

# Map CodeHaluEval halu_type strings → our HallucinationType enum
HALU_TYPE_MAP = {
    'physical_constraint_hallucination': HallucinationType.LOGIC,
    'logic_breakdown':                   HallucinationType.LOGIC,
    'logic_deviation':                   HallucinationType.LOGIC,
    'calculate_boundary_hallucination':  HallucinationType.LOGIC,
    'identification_hallucination':      HallucinationType.NAMING,
    'structural_access_hallucination':   HallucinationType.MAPPING,
    'data_compliance_hallucination':     HallucinationType.MAPPING,
    'external_source_hallucination':     HallucinationType.RESOURCE,
}


def _parse_value(s: str):
    """Parse a competitive-programming input/output string into a Python value."""
    s = s.strip()
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        pass
    lines = s.splitlines()
    if len(lines) == 1:
        tokens = s.split()
        if len(tokens) == 1:
            for cast in (int, float):
                try: return cast(s)
                except ValueError: pass
            return s
        parsed = []
        for t in tokens:
            for cast in (int, float):
                try: parsed.append(cast(t)); break
                except ValueError: pass
            else:
                parsed.append(t)
        return parsed
    return [_parse_value(line) for line in lines]


def build_test_code(fn_name: str, input_str: str, output_str: str) -> str:
    """Build an executable assertion from a function name and one input/output pair.

    The generated code is expected to define `fn_name` (or 'solve') as a callable.
    """
    fn = (fn_name or 'solve').strip()
    input_val   = _parse_value(input_str)
    expected_val = _parse_value(output_str)

    if isinstance(input_val, list):
        args_repr = ', '.join(repr(v) for v in input_val)
    else:
        args_repr = repr(input_val)

    return (
        f"_result   = {fn}({args_repr})\n"
        f"_expected = {repr(expected_val)}\n"
        f"assert _result == _expected, "
        f"f'Expected {{_expected!r}}, got {{_result!r}}'"
    )


def extract_task(sample: dict) -> dict:
    """Extract a normalised task dict from a CodeHaluEval sample.

    CodeHaluEval schema:
        id, task_id, question, solutions (list[str]),
        input, output, fn_name, halu_type, starter_code, ...
    """
    # Use solutions as repo context (each solution is a reference implementation)
    solutions = sample.get('solutions') or []
    if isinstance(solutions, str):
        solutions = [solutions]
    repo_files = {f'solution_{i}.py': sol for i, sol in enumerate(solutions) if sol}
    if not repo_files:
        repo_files = {'placeholder.py': '# No reference solution provided'}

    fn_name   = sample.get('fn_name') or 'solve'
    test_code = build_test_code(fn_name, sample.get('input', ''), sample.get('output', ''))

    return {
        'task_id':          str(sample.get('id', sample.get('task_id', '?'))),
        'query':            sample['question'],
        'repo_files':       repo_files,
        'test_code':        test_code,
        'fn_name':          fn_name,
        'hallucination_type': HALU_TYPE_MAP.get(
            (sample.get('halu_type') or '').lower().replace(' ', '_')
        ),
    }


# Smoke test — run after loading dataset
sample = ds[0]
task   = extract_task(sample)
print('task_id  :', task['task_id'])
print('fn_name  :', task['fn_name'])
print('halu_type:', task['hallucination_type'])
print('repo_files keys:', list(task['repo_files'].keys()))
print('test_code preview:\n', task['test_code'])
print()
print('query preview:\n', task['query'][:200])

## 2. Load Code Generation Model (DeepSeek-Coder 1.3B)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-instruct'

print(f'Loading {MODEL_NAME}...')
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
gen_model.eval()
print(f'Model loaded on {next(gen_model.parameters()).device}')
print(f'Model size: {sum(p.numel() for p in gen_model.parameters()) / 1e6:.0f}M params')

In [ ]:
def build_prompt(query: str, fn_name: str, context_chunks: list) -> str:
    """Assemble a prompt that asks DeepSeek-Coder to write a specific function."""
    ctx = '\n\n'.join(context_chunks) if context_chunks else '# (no reference context)'
    fn  = fn_name or 'solve'
    return (
        f"# Reference implementations for context:\n\n{ctx}\n\n"
        f"# Task:\n# {query}\n\n"
        f"# Write a Python function named `{fn}` that solves the task above.\n"
        f"# Only output the function definition — no main block, no explanation.\n\n"
        f"def {fn}("
    )


def generate_code(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate Python code from a prompt using DeepSeek-Coder-1.3B.

    Temperature 0.6: low enough for coherent code, high enough for
    diverse outputs across different context subsets (essential for
    getting both PASS and FAIL outcomes to form contrastive triplets).
    """
    inputs = gen_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    # Strip the prompt tokens — only decode newly generated tokens
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    generated  = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)

    # The prompt ends with "def fn_name(" — prepend it so code is complete
    fn_name_start = prompt.split('\ndef ')[1].split('(')[0] if '\ndef ' in prompt else 'solve'
    return f"def {fn_name_start}(" + generated


# Quick smoke test (will fail until model is loaded — that's expected)
try:
    test_prompt = build_prompt("Return x+1", "increment", [])
    print("Prompt preview:")
    print(test_prompt[:200])
except Exception as e:
    print(f"build_prompt OK; generate_code needs model: {e}")

## 3. Generate Contrastive Triplets

In [ ]:
import json
from haluguard.data_pipeline import _triplet_to_dict

N_TASKS   = 500  # start with 10 to validate end-to-end, then scale to 500
N_SUBSETS = 6
SEED      = 42

already_done_ids = set()
if TRIPLETS_PATH.exists():
    with TRIPLETS_PATH.open() as f:
        for line in f:
            if line.strip():
                already_done_ids.add(json.loads(line).get('task_id', ''))
print(f'Already processed {len(already_done_ids)} task IDs, '
      f'{count_jsonl(TRIPLETS_PATH)} triplets on disk')

# ds is now a plain list — slice directly (no .select())
tasks_to_run = [
    s for s in ds[:N_TASKS]
    if str(s.get('id', s.get('task_id', ''))) not in already_done_ids
]
print(f'Tasks to process this run: {len(tasks_to_run)}')

new_triplets = 0
for i, sample in enumerate(tqdm(tasks_to_run, desc='Generating triplets')):
    task = extract_task(sample)
    fn_name = task['fn_name']

    def _gen(prompt: str, _fn=fn_name, _task=task) -> str:
        return generate_code(
            build_prompt(_task['query'], _fn, prompt.split('\n\n') if prompt else [])
        )

    try:
        triplets = create_triplets_from_task(
            task_id=task['task_id'],
            query=task['query'],
            repo_files=task['repo_files'],
            test_code=task['test_code'],
            generate_fn=_gen,
            n_subsets=N_SUBSETS,
            seed=SEED + i,
        )
        for triplet in triplets:
            append_jsonl(_triplet_to_dict(triplet), TRIPLETS_PATH)
            new_triplets += 1
    except Exception as e:
        print(f'  Task {task["task_id"]} failed: {e}')
        continue

print(f'\nDone. Added {new_triplets} new triplets. '
      f'Total on disk: {count_jsonl(TRIPLETS_PATH)}')

## 4. Analyse Triplets

In [ ]:
if TRIPLETS_PATH.exists():
    triplets = load_triplets(TRIPLETS_PATH)
    summary  = summarise_triplets(triplets)
    print(f'Total triplets: {summary["total"]}')
    print('By hallucination type:')
    for t, count in summary['by_hallucination_type'].items():
        pct = 100 * count / max(summary['total'], 1)
        print(f'  {t:10s}: {count:5d} ({pct:.1f}%)')
    print(f'Unknown type: {summary["unknown"]}')

    if summary['total'] < 100:
        print('\nWARNING: fewer than 100 triplets — increase N_TASKS or check that tasks produce failures.')
    elif summary['total'] >= 1000:
        print('\nReady for training! Run notebook 02 next.')
else:
    print('No triplets yet — run the generation cell above first.')

In [ ]:
# Save progress to GitHub
import subprocess
os.chdir(REPO_DIR)
!git add -A
!git commit -m 'chore: update after notebook 01 data generation'
!git push